In [1]:
import os
import pandas as pd

# ============================================================
# Config
# ============================================================
# BASE_DIR 자동 탐색:
# - 현재 폴더에 result_1이 있으면 BASE_DIR="."
# - 아니면 ./7_prediction_avg/result_1이 있으면 BASE_DIR="./7_prediction_avg"
if os.path.isdir("./result_1"):
    BASE_DIR = "."
elif os.path.isdir("./7_prediction_avg/result_1"):
    BASE_DIR = "./7_prediction_avg"
else:
    raise FileNotFoundError(
        "result_1 폴더를 찾지 못했습니다.\n"
        "1) 7_prediction_avg 폴더 안에서 실행하거나\n"
        "2) 상위 폴더에서 실행한다면 ./7_prediction_avg/result_1 이 존재해야 합니다."
    )

RESULT_DIRS = [f"result_{i}" for i in range(1, 6)]

# 평균 낼 파일 종류(각 result_k 폴더 안에 results_*_k.csv 형태로 존재)
BASE_FILES = [
    "results_O.csv",
    "results_F.csv",
    "results_OF.csv",
    "results_OFR.csv",
    "results_OFR_trials.csv",
]

OUT_DIR = os.path.join(BASE_DIR, "avg_results")
os.makedirs(OUT_DIR, exist_ok=True)

# 정렬 고정 모델 순서
MODEL_ORDER = ["XGBoost", "LightGBM", "RF", "Logit", "FFMLP"]

# groupby 키(고정)
GROUP_KEYS = ["SET", "DAG", "MODEL", "K_EDGE", "N_FEAT"]

# 최종 출력 컬럼(고정)
FINAL_COLS = ["SET", "DAG", "MODEL", "K_EDGE", "N_FEAT", "AUROC", "AUPRC", "F1", "Brier", "ECE"]


# ============================================================
# Helpers
# ============================================================
def file_for_result(base_file: str, idx: int) -> str:
    """
    base_file: 'results_F.csv'
    idx=1..5 -> 'results_F_1.csv' ... 'results_F_5.csv'
    """
    stem, ext = os.path.splitext(base_file)
    return f"{stem}_{idx}{ext}"


def require_columns(df: pd.DataFrame, required_cols: list, context: str):
    missing = [c for c in required_cols if c not in df.columns]
    if missing:
        raise ValueError(f"[{context}] 필수 컬럼 누락: {missing}\n현재 컬럼: {list(df.columns)}")


def average_one_kind(base_file: str) -> pd.DataFrame:
    """
    base_file 종류(예: results_F.csv)에 대해 result_1~5의 CSV를 읽어 평균낸 DF를 반환
    """
    dfs = []
    for idx, rdir in enumerate(RESULT_DIRS, start=1):
        fname = file_for_result(base_file, idx)
        path = os.path.join(BASE_DIR, rdir, fname)
        if not os.path.exists(path):
            raise FileNotFoundError(f"Missing file: {path}")

        df = pd.read_csv(path)

        # FEATURE_KEY 제거(있으면 무조건 제거)
        if "FEATURE_KEY" in df.columns:
            df = df.drop(columns=["FEATURE_KEY"])

        dfs.append(df)

    # 컬럼 구조 동일성 체크 (FEATURE_KEY 제거 이후 기준)
    cols0 = dfs[0].columns.tolist()
    for idx, df in enumerate(dfs, start=1):
        if df.columns.tolist() != cols0:
            raise ValueError(
                f"[{base_file}] 컬럼 구조가 run마다 다릅니다: result_{idx}\n"
                f"result_1 columns: {cols0}\n"
                f"result_{idx} columns: {df.columns.tolist()}"
            )

    df_all = pd.concat(dfs, ignore_index=True)

    # 필수 컬럼 존재 확인
    require_columns(df_all, GROUP_KEYS, context=f"{base_file} (group keys)")
    require_columns(df_all, ["AUROC", "AUPRC", "F1", "Brier", "ECE"], context=f"{base_file} (metrics)")

    # 숫자형으로 확실히 캐스팅(혹시 문자열로 들어온 경우 방지)
    for c in ["K_EDGE", "N_FEAT", "AUROC", "AUPRC", "F1", "Brier", "ECE"]:
        df_all[c] = pd.to_numeric(df_all[c], errors="coerce")

    # groupby mean (키는 고정)
    metric_cols = ["AUROC", "AUPRC", "F1", "Brier", "ECE"]
    avg_df = (
        df_all
        .groupby(GROUP_KEYS, dropna=False, as_index=False)[metric_cols]
        .mean()
    )

    # 정렬: DAG -> MODEL(고정순서) -> N_FEAT
    avg_df["MODEL"] = pd.Categorical(avg_df["MODEL"], categories=MODEL_ORDER, ordered=True)
    avg_df = avg_df.sort_values(by=["DAG", "MODEL", "N_FEAT"], na_position="last").reset_index(drop=True)

    # 최종 컬럼/순서 고정(정확히 10개)
    require_columns(avg_df, FINAL_COLS, context=f"{base_file} (final columns)")
    avg_df = avg_df[FINAL_COLS]

    return avg_df


def main():
    print(f"[INFO] BASE_DIR = {BASE_DIR}")
    print(f"[INFO] OUT_DIR  = {OUT_DIR}")

    for bf in BASE_FILES:
        print(f"\nProcessing: {bf}")
        avg_df = average_one_kind(bf)

        out_name = bf.replace(".csv", "_AVG.csv")
        out_path = os.path.join(OUT_DIR, out_name)
        avg_df.to_csv(out_path, index=False)
        print(f"[SAVED] {out_path}  (rows={len(avg_df)})")

    print("\nAll done.")


if __name__ == "__main__":
    main()

[INFO] BASE_DIR = .
[INFO] OUT_DIR  = .\avg_results

Processing: results_O.csv
[SAVED] .\avg_results\results_O_AVG.csv  (rows=20)

Processing: results_F.csv
[SAVED] .\avg_results\results_F_AVG.csv  (rows=510)

Processing: results_OF.csv
[SAVED] .\avg_results\results_OF_AVG.csv  (rows=510)

Processing: results_OFR.csv
[SAVED] .\avg_results\results_OFR_AVG.csv  (rows=120)

Processing: results_OFR_trials.csv
[SAVED] .\avg_results\results_OFR_trials_AVG.csv  (rows=120)

All done.
